In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small/checkpoints/post_cell_27_try_4.pickle

In [ ]:
%%cudf.pandas.profile
### cell 28 ###

# 1. Standardize question prefixes and answer labels across all DataFrames
orig_questions = [
    'Which of the following hosted notebook products do you use on a regular basis?',
    'Which of the following hosted notebooks have you used at work or school in the last 5 years?'
]
new_question = 'Do you use any of the following hosted notebook products?'
label_replacements = {
    'Google Colab ':             'Colab Notebooks',
    'Kaggle Notebooks (Kernels) ': 'Kaggle Notebooks',
    'Kaggle Kernels':             'Kaggle Notebooks'
}

dfs_to_clean = [
    responses_df_2017,
    responses_df_2018_cell10,
    responses_df_2019_cell10,
    responses_df_2020,
    responses_df_2021,
    responses_df_2022_cell10
]
for df in dfs_to_clean:
    cols = df.columns
    for oq in orig_questions:
        cols = cols.str.replace(oq, new_question, regex=False)
    for old, new in label_replacements.items():
        cols = cols.str.replace(old, new, regex=False)
    df.columns = cols

# 2–5. Melt, count, compute percentages in long form
year_sources = [
    (2018, responses_df_2018_cell10),
    (2019, responses_df_2019_cell10),
    (2020, responses_df_2020),
    (2021, responses_df_2021),
    (2022, responses_df_2022_cell10)
]
counts_list = []
for yr, df_src in year_sources:
    # select only the notebook question columns
    subset = df_src.loc[:, df_src.columns.str.contains(new_question)]
    # melt into long form
    melted = subset.melt(value_name='response')
    # drop missing and extract answer label
    melted = melted.dropna(subset=['response'])
    melted['response'] = (
        melted['variable']
          .str.split('-', 1)
          .str.get(1)
          .str.strip()
    )
    # group by year and answer
    year_counts = (
        melted.assign(year=str(yr))
              .groupby(['year', 'response'])
              .size()
              .reset_index(name='count')
    )
    counts_list.append(year_counts)

# concatenate yearly counts
counts_df = pd.concat(counts_list, ignore_index=True)
# compute totals and percentages
totals = counts_df.groupby('year')['count'].sum().reset_index(name='total')
percentages_df = counts_df.merge(totals, on='year')
percentages_df['percent'] = percentages_df['count'].div(percentages_df['total']).mul(100)

# prepare for plotting: filter answers of interest and rename
df_cell40 = (
    percentages_df
      .loc[percentages_df['response'].isin(['None', 'Kaggle Notebooks', 'Colab Notebooks']),
           ['year', 'response', 'percent']]
      .rename(columns={'response': ''})
)
# inspect
df_cell40.info()